# Macro Regime Analysis

This project studies how stocks, bonds, gold, and commodities perform when economic growth and inflation are rising or falling.

## Import Libraries

These libraries are used to download data, calculate returns, and create charts.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pandas_datareader import data as web

## Project Settings

SPY represents equities, TLT represents long-term government bonds, GLD represents gold, and DBC represents commodities.

In [ ]:
asset_names = {
    "SPY": "Equities",
    "TLT": "Bonds",
    "GLD": "Gold",
    "DBC": "Commodities"
}

asset_tickers = list(asset_names.keys())

start_date = "2007-01-01"
end_date = pd.Timestamp.today().strftime("%Y-%m-%d")

trading_months = 12
risk_free_rate = 0.02

## Market Data Collection

Historical adjusted prices are downloaded and converted into monthly prices and returns.

In [ ]:
market_data = yf.download(
    asset_tickers,
    start=start_date,
    end=end_date,
    auto_adjust=True,
    progress=False
)

daily_prices = market_data["Close"].copy()
daily_prices = daily_prices[asset_tickers]

daily_prices.index = daily_prices.index.to_period("M")
monthly_prices = daily_prices.groupby(level=0).last()
monthly_returns = monthly_prices.pct_change().dropna()

monthly_prices.tail()

## Economic Data Collection

CPI measures consumer prices and INDPRO measures US industrial production. The data is downloaded from the Federal Reserve Economic Data database.

In [ ]:
fred_series = ["CPIAUCSL", "INDPRO"]

macro_data = web.DataReader(
    fred_series,
    "fred",
    start_date,
    end_date
)

macro_data.columns = ["CPI", "Industrial Production"]
macro_data.index = macro_data.index.to_period("M")
macro_data = macro_data.dropna()

macro_data.tail()

## Economic Indicator Calculation

Annual inflation and industrial production growth are calculated. A three-month change shows whether each indicator is accelerating or slowing.

In [ ]:
macro_data["Inflation"] = macro_data["CPI"].pct_change(12) * 100
macro_data["Growth"] = macro_data["Industrial Production"].pct_change(12) * 100

macro_data["Inflation Trend"] = macro_data["Inflation"].diff(3)
macro_data["Growth Trend"] = macro_data["Growth"].diff(3)

macro_data = macro_data.dropna()

macro_data[["Inflation", "Growth", "Inflation Trend", "Growth Trend"]].tail()

## Economic Regime Classification

Each month is classified using the direction of growth and inflation.

In [ ]:
conditions = [
    (macro_data["Growth Trend"] >= 0) & (macro_data["Inflation Trend"] < 0),
    (macro_data["Growth Trend"] >= 0) & (macro_data["Inflation Trend"] >= 0),
    (macro_data["Growth Trend"] < 0) & (macro_data["Inflation Trend"] >= 0),
    (macro_data["Growth Trend"] < 0) & (macro_data["Inflation Trend"] < 0)
]

regimes = ["Goldilocks", "Reflation", "Stagflation", "Deflation"]

macro_data["Regime"] = np.select(conditions, regimes, default="Unknown")

macro_data[["Inflation", "Growth", "Regime"]].tail(12)

## Combine Market and Economic Data

Monthly asset returns are combined with the economic regime for the same month.

In [ ]:
analysis_data = monthly_returns.join(
    macro_data[["Inflation", "Growth", "Inflation Trend", "Growth Trend", "Regime"]],
    how="inner"
)

analysis_data = analysis_data.dropna()

analysis_data.tail()

## Regime Summary

This counts how many months occurred in each regime.

In [ ]:
regime_counts = analysis_data["Regime"].value_counts()

regime_summary = pd.DataFrame({
    "Months": regime_counts,
    "Percentage": regime_counts / regime_counts.sum()
})

regime_summary

## Asset Performance by Regime

Average monthly returns are annualised to make the regimes easier to compare.

In [ ]:
regime_returns = (
    analysis_data.groupby("Regime")[asset_tickers].mean()
    * trading_months
)

regime_returns = regime_returns.rename(columns=asset_names)

print("--- Annualised Average Returns by Regime ---")
display(regime_returns.style.format("{:.2%}"))

## Asset Risk by Regime

Annualised volatility shows how unstable each asset was in each regime.

In [ ]:
regime_volatility = (
    analysis_data.groupby("Regime")[asset_tickers].std()
    * np.sqrt(trading_months)
)

regime_volatility = regime_volatility.rename(columns=asset_names)

print("--- Annualised Volatility by Regime ---")
display(regime_volatility.style.format("{:.2%}"))

## Current Economic Regime

The most recent available inflation and production data is used to identify the latest regime.

In [ ]:
latest_data = macro_data.iloc[-1]

print("--- Current Economic Regime ---")
print(f"Date: {macro_data.index[-1]}")
print(f"Regime: {latest_data['Regime']}")
print(f"Inflation: {latest_data['Inflation']:.2f}%")
print(f"Industrial Production Growth: {latest_data['Growth']:.2f}%")

## Asset Growth Visualisation

This chart shows the growth of $1 invested in each asset.

In [ ]:
asset_growth = (1 + analysis_data[asset_tickers]).cumprod()
asset_growth = asset_growth.rename(columns=asset_names)

asset_growth.plot(figsize=(12, 6))
plt.title("Growth of $1 by Asset Class")
plt.ylabel("Portfolio Value")
plt.xlabel("Date")
plt.grid(alpha=0.3)
plt.show()

## Regime Return Heatmap

The heatmap makes it easier to identify which assets performed best and worst in each regime.

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(
    regime_returns,
    annot=True,
    fmt=".1%",
    cmap="RdYlGn",
    center=0
)
plt.title("Annualised Asset Returns by Economic Regime")
plt.xlabel("Asset Class")
plt.ylabel("Economic Regime")
plt.show()

## Regime Frequency Visualisation

This chart shows how often each regime appeared in the historical sample.

In [ ]:
plt.figure(figsize=(9, 5))
regime_counts.reindex(regimes).plot(kind="bar", color="#4472C4")
plt.title("Number of Months in Each Economic Regime")
plt.ylabel("Months")
plt.xlabel("Economic Regime")
plt.xticks(rotation=0)
plt.show()

## Illustrative Regime Portfolio

The portfolio uses different asset weights for each regime. The regime is shifted by one month so the strategy only uses information from the previous month.

In [ ]:
regime_weights = {
    "Goldilocks": {"SPY": 0.50, "TLT": 0.25, "GLD": 0.15, "DBC": 0.10},
    "Reflation": {"SPY": 0.35, "TLT": 0.10, "GLD": 0.20, "DBC": 0.35},
    "Stagflation": {"SPY": 0.15, "TLT": 0.10, "GLD": 0.35, "DBC": 0.40},
    "Deflation": {"SPY": 0.15, "TLT": 0.55, "GLD": 0.25, "DBC": 0.05}
}

analysis_data["Signal Regime"] = analysis_data["Regime"].shift(1)

strategy_returns = []

for date, row in analysis_data.iterrows():
    signal_regime = row["Signal Regime"]

    if signal_regime in regime_weights:
        weights = regime_weights[signal_regime]
        monthly_return = sum(row[ticker] * weights[ticker] for ticker in asset_tickers)
    else:
        monthly_return = np.nan

    strategy_returns.append(monthly_return)

analysis_data["Regime Portfolio"] = strategy_returns
analysis_data["Equal Weight"] = analysis_data[asset_tickers].mean(axis=1)

strategy_data = analysis_data[["Regime Portfolio", "Equal Weight", "SPY"]].dropna()

strategy_data.head()

## Strategy Performance

The regime portfolio is compared with SPY and a portfolio that holds equal weights in all four assets.

In [ ]:
def calculate_metrics(return_series):
    annual_return = (1 + return_series).prod() ** (trading_months / len(return_series)) - 1
    volatility = return_series.std() * np.sqrt(trading_months)
    sharpe_ratio = (annual_return - risk_free_rate) / volatility

    growth = (1 + return_series).cumprod()
    drawdown = growth / growth.cummax() - 1
    maximum_drawdown = drawdown.min()

    return annual_return, volatility, sharpe_ratio, maximum_drawdown


performance_results = {}

for column in strategy_data.columns:
    performance_results[column] = calculate_metrics(strategy_data[column])

performance_summary = pd.DataFrame(
    performance_results,
    index=["Annual Return", "Volatility", "Sharpe Ratio", "Maximum Drawdown"]
).T

performance_summary

## Strategy Growth Visualisation

The final chart compares the historical growth of the three portfolios.

In [ ]:
strategy_growth = (1 + strategy_data).cumprod()

strategy_growth.plot(figsize=(12, 6))
plt.title("Regime Portfolio vs Equal Weight and SPY")
plt.ylabel("Growth of $1")
plt.xlabel("Date")
plt.grid(alpha=0.3)
plt.show()

## Conclusion

This project shows how inflation and economic growth can be converted into measurable economic regimes.

It compares asset returns and risk across regimes, identifies the latest regime, and tests a simple portfolio using lagged information.

The main learning is that an investment idea should include an economic explanation, clear rules, historical testing, and an honest discussion of limitations.